In [28]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import os
from shapely.geometry import Point, Polygon, LineString
from datetime import timedelta
from tqdm import tqdm

In [22]:
region = gpd.read_file(r'C:\Users\TJ002\Desktop\code\Cal_code_data\NWP_orbit_processing\Arctic_Canada_North.shp')
s1_metadata_path = r"E:\NWP\Sentinel1\Metadata\filtered_GRD_files\2023\sentinel1_2023_all_GRD_filtered.csv"
cs2_base_dir = r"E:\NWP\CS2\2023"

output_path =  r"E:\NWP\CS2_S1_matched"


In [23]:
def clip_nc_by_region(nc_path, region_gdf):
    """
    读取 CryoSat-2 .nc 文件，只保留落在 region 范围内的点位
    """
    try:
        ds = xr.open_dataset(nc_path)
        lats = ds['lat_01'].values
        lons = ds['lon_01'].values

        # 构建点的 GeoSeries
        points = gpd.GeoSeries([Point(lon, lat) for lon, lat in zip(lons, lats)], crs="EPSG:4326")
        
        # region_gdf 已经是 GeoDataFrame，可能是多多边形，做 union
        region_union = region_gdf.unary_union
        
        # 使用 contains 向量判断哪些点在区域内
        mask = points.within(region_union)
        
        # 若无点落在区域中，返回 None
        if not mask.any():
            return None

        return {
            "lat": lats[mask],
            "lon": lons[mask],
            "time": ds['time_cor_01'].values[mask],
            "path": nc_path
        }
    except Exception as e:
        print(f"❌ 读取失败 {nc_path}: {e}")
        return None


def build_s1_polygon(row):
    """
    根据 Sentinel-1 元数据构建足迹多边形
    """
    coords = [
        (row['Near Start Lon'], row['Near Start Lat']),
        (row['Far Start Lon'], row['Far Start Lat']),
        (row['Far End Lon'], row['Far End Lat']),
        (row['Near End Lon'], row['Near End Lat']),
        (row['Near Start Lon'], row['Near Start Lat'])  # 闭合
    ]
    return Polygon(coords)

In [24]:
s1_df = pd.read_csv(r"E:\NWP\Sentinel1\Metadata\filtered_GRD_files\2023\sentinel1_2023_all_GRD_filtered.csv", parse_dates=["Start Time", "End Time"])
# 移除时区信息，避免与 CryoSat-2 时间冲突
s1_df["Start Time"] = s1_df["Start Time"].dt.tz_localize(None)
s1_df["End Time"] = s1_df["End Time"].dt.tz_localize(None)
# -------------------------
# Clip CryoSat-2 by region first
# -------------------------
cs2_clipped_summary = []

year_path = r"E:\NWP\CS2\2023"  # Change to your CS2 root folder


for root, _, files in os.walk(year_path):
    for f in files:
        if f.endswith(".nc"):
            full_path = os.path.join(root, f)
            try:
                clipped = clip_nc_by_region(full_path, region)
                if clipped is not None:
                    times = pd.to_datetime(clipped["time"])
                    cs2_clipped_summary.append({
                        "cs2_path": full_path,
                        "cs2_start": times[0],
                        "cs2_end": times[-1],
                        "lat": clipped["lat"],
                        "lon": clipped["lon"],
                        "time": clipped["time"]
                    })
            except Exception as e:
                print(f"❌ 读取失败 {full_path}: {e}")

print(f"✅ 区域内 CS2 文件数：{len(cs2_clipped_summary)}")
# -------------------------

✅ 区域内 CS2 文件数：2641


In [25]:
time_matches = []
    
for _, s1_row in tqdm(s1_df.iterrows(), total=len(s1_df), desc="时间匹配"):
    s1_start, s1_end = s1_row["Start Time"], s1_row["End Time"]
    s1_lon, s1_lat = s1_row["Center Lon"], s1_row["Center Lat"]
    
    # 构造 ±2 小时窗口
    time_window_start = s1_start - timedelta(hours=2)
    time_window_end = s1_end + timedelta(hours=2)
    
    for cs2 in cs2_clipped_summary:
        # 判断是否时间重叠（±2小时窗口）
        if (cs2["cs2_end"] >= time_window_start) and (cs2["cs2_start"] <= time_window_end):
            # 简易空间粗筛（S1 中心点在 CS2 点 bbox 内）
            if (cs2["lat"].min() <= s1_lat <= cs2["lat"].max()) and \
                (cs2["lon"].min() <= s1_lon <= cs2["lon"].max()):
                time_matches.append({
                    "sceneName": s1_row["Granule Name"],
                    "s1_start": s1_start,
                    "s1_end": s1_end,
                    "s1_lon": s1_lon,
                    "s1_lat": s1_lat,
                    "cs2_path": cs2["cs2_path"],
                    "cs2_start": cs2["cs2_start"],
                    "cs2_end": cs2["cs2_end"]
                })
    
print(time_matches)
print(len(time_matches))

时间匹配: 100%|██████████| 2165/2165 [00:00<00:00, 3141.17it/s]

[{'sceneName': 'S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83', 's1_start': Timestamp('2023-06-30 13:47:49'), 's1_end': Timestamp('2023-06-30 13:48:53'), 's1_lon': -93.0747, 's1_lat': 79.5309, 'cs2_path': 'E:\\NWP\\CS2\\2023\\CS_OFFL_SIR_SIN_2__20230630T115732_20230630T120238_E001.nc', 'cs2_start': Timestamp('2023-06-30 12:00:56.449635968'), 'cs2_end': Timestamp('2023-06-30 12:03:14.267412992')}, {'sceneName': 'S1A_EW_GRDM_1SDH_20230629T144524_20230629T144629_049202_05EA9F_5AF6', 's1_start': Timestamp('2023-06-29 14:45:24'), 's1_end': Timestamp('2023-06-29 14:46:29'), 's1_lon': -107.4076, 's1_lat': 79.5672, 'cs2_path': 'E:\\NWP\\CS2\\2023\\CS_OFFL_SIR_SIN_2__20230629T124858_20230629T125249_E001.nc', 'cs2_start': Timestamp('2023-06-29 12:51:30.832275072'), 'cs2_end': Timestamp('2023-06-29 12:53:26.108185088')}, {'sceneName': 'S1A_EW_GRDM_1SDH_20230629T130747_20230629T130819_049201_05EA98_1A8E', 's1_start': Timestamp('2023-06-29 13:07:47'), 's1_end': Timestamp('2023

In [26]:
spatial_matches = []
    
for match in tqdm(time_matches, desc="空间匹配"):
    # 找到对应的 Sentinel-1 行信息
    s1_row = s1_df[s1_df["Granule Name"] == match["sceneName"]].iloc[0]
    
    # 构建 S1 足迹多边形
    s1_poly = build_s1_polygon(s1_row)
    
    # 打开对应 CS2 文件
    try:
        ds = xr.open_dataset(match["cs2_path"])
        lat = ds["lat_01"].values
        lon = ds["lon_01"].values
        time = pd.to_datetime(ds["time_cor_01"].values)
        
        # 构建 CS2 轨迹线
        cs2_points = [Point(lo, la) for lo, la in zip(lon, lat)]
        if len(cs2_points) >= 2:  # 至少需要两个点才能构成线
            cs2_line = LineString(cs2_points)
            
            # 判断轨迹线是否与多边形相交
            if cs2_line.intersects(s1_poly):
                # 第三步（可选）：CS2 有效穿越点个数统计
                # 找出同时满足时间和空间条件的点
                time_window_start = match["s1_start"] - timedelta(hours=2)
                time_window_end = match["s1_end"] + timedelta(hours=2)
                
                valid_points = []
                for lo, la, t in zip(lon, lat, time):
                    if time_window_start <= t <= time_window_end:
                        pt = Point(lo, la)
                        if s1_poly.contains(pt):
                            valid_points.append({
                                "cs2_time": t,
                                "cs2_lat": la,
                                "cs2_lon": lo
                            })
                
                if valid_points:  # 只有有效点才添加到结果中
                    valid_times = [p["cs2_time"] for p in valid_points]
                    spatial_matches.append({
                        "sceneName": match["sceneName"],
                        "s1_start": match["s1_start"],
                        "s1_end": match["s1_end"],
                        "s1_url": s1_row["URL"] if "URL" in s1_row else "",
                        "cs2_path": match["cs2_path"],
                        "cs2_time_min": min(valid_times),
                        "cs2_time_max": max(valid_times),
                        "num_matched_points": len(valid_points)
                    })
    except Exception as e:
        print(f"❌ 处理失败 {match['cs2_path']}: {e}")
    
    print(f"✅ 空间匹配成功数：{len(spatial_matches)}")

空间匹配:   8%|▊         | 10/121 [00:00<00:02, 46.57it/s]

✅ 空间匹配成功数：1
✅ 空间匹配成功数：2
✅ 空间匹配成功数：2
✅ 空间匹配成功数：3
✅ 空间匹配成功数：4
✅ 空间匹配成功数：5
✅ 空间匹配成功数：6
✅ 空间匹配成功数：7
✅ 空间匹配成功数：8
✅ 空间匹配成功数：9


空间匹配:  12%|█▏        | 15/121 [00:00<00:02, 46.37it/s]

✅ 空间匹配成功数：10
✅ 空间匹配成功数：10
✅ 空间匹配成功数：11
✅ 空间匹配成功数：12
✅ 空间匹配成功数：13
✅ 空间匹配成功数：14
✅ 空间匹配成功数：15
✅ 空间匹配成功数：16
✅ 空间匹配成功数：17


空间匹配:  21%|██        | 25/121 [00:00<00:02, 41.58it/s]

...

✅ 空间匹配成功数：85
✅ 空间匹配成功数：86
✅ 空间匹配成功数：87
✅ 空间匹配成功数：88
✅ 空间匹配成功数：88
✅ 空间匹配成功数：88
✅ 空间匹配成功数：89
✅ 空间匹配成功数：90
✅ 空间匹配成功数：91
✅ 空间匹配成功数：92
✅ 空间匹配成功数：93


空间匹配:  91%|█████████ | 110/121 [00:02<00:00, 49.39it/s]

✅ 空间匹配成功数：94
✅ 空间匹配成功数：95
✅ 空间匹配成功数：96
✅ 空间匹配成功数：97
✅ 空间匹配成功数：98
✅ 空间匹配成功数：99
✅ 空间匹配成功数：100
✅ 空间匹配成功数：101
✅ 空间匹配成功数：102
✅ 空间匹配成功数：103
✅ 空间匹配成功数：104


空间匹配: 100%|██████████| 121/121 [00:02<00:00, 46.95it/s]

✅ 空间匹配成功数：105
✅ 空间匹配成功数：106
✅ 空间匹配成功数：107
✅ 空间匹配成功数：108
✅ 空间匹配成功数：109
✅ 空间匹配成功数：110
✅ 空间匹配成功数：111
✅ 空间匹配成功数：112


In [27]:
matches_df = pd.DataFrame(spatial_matches)
print(matches_df)
result_df = pd.DataFrame(spatial_matches)
os.makedirs(os.path.dirname(output_path), exist_ok=True)
matches_df.to_csv("time_matched_pairs.csv", index=False)
print(f"✅ 结果已保存至：{output_path}")

                                             sceneName            s1_start  \
0    S1A_EW_GRDM_1SDH_20230630T134749_20230630T1348... 2023-06-30 13:47:49   
1    S1A_EW_GRDM_1SDH_20230629T144524_20230629T1446... 2023-06-29 14:45:24   
2    S1A_EW_GRDM_1SDH_20230629T130642_20230629T1307... 2023-06-29 13:06:42   
3    S1A_EW_GRDM_1SDH_20230628T140418_20230628T1405... 2023-06-28 14:04:18   
4    S1A_IW_GRDH_1SDH_20230628T122528_20230628T1225... 2023-06-28 12:25:28   
..                                                 ...                 ...   
107  S1A_IW_GRDH_1SDH_20230703T141341_20230703T1414... 2023-07-03 14:13:41   
108  S1A_IW_GRDH_1SDH_20230703T141316_20230703T1413... 2023-07-03 14:13:16   
109  S1A_EW_GRDM_1SDH_20230703T123501_20230703T1236... 2023-07-03 12:35:01   
110  S1A_EW_GRDM_1SDH_20230701T142859_20230701T1430... 2023-07-01 14:28:59   
111  S1A_EW_GRDM_1SDH_20230701T125014_20230701T1251... 2023-07-01 12:50:14   

                 s1_end                                        